# SmoothQuant W8A8

SmoothQuant is a post-training method for W8A8 quantization. The official README describes it as "training-free, accuracy-preserving" and says it migrates activation outlier difficulty into weights.

Sources:
- Paper: https://arxiv.org/abs/2211.10438
- Official code: https://github.com/mit-han-lab/smoothquant
- Intel Neural Compressor SmoothQuant docs: https://intel.github.io/neural-compressor/latest/docs/source/smooth_quant.html

Practical note: the official real-INT8 PyTorch path is focused on OPT models and custom INT8 kernels. This notebook gives you two paths:
1. Load an official already-exported SmoothQuant OPT checkpoint from Hugging Face and save it locally.
2. Run the official smoothing plus fake W8A8 path to evaluate your own OPT variant.

In [ ]:
!pip -q install -U transformers==4.36.0 accelerate datasets zstandard sentencepiece safetensors
!git clone -q https://github.com/mit-han-lab/smoothquant /content/smoothquant || true
%cd /content/smoothquant
!pip -q install -e .

In [ ]:
import json
import os
import time
from pathlib import Path

import torch
from datasets import load_dataset
from transformers import AutoTokenizer
from transformers.models.opt.modeling_opt import OPTForCausalLM
from smoothquant.opt import Int8OPTForCausalLM
from smoothquant.smooth import smooth_lm
from smoothquant.fake_quant import quantize_opt


def bytes_to_gib(n):
    return n / (1024 ** 3)


def directory_size_bytes(path):
    path = Path(path)
    if not path.exists():
        return 0
    return sum(p.stat().st_size for p in path.rglob("*") if p.is_file())


def save_metrics(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(json.dumps(payload, indent=2))


In [ ]:
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
SMOOTHQUANT_HF_ID = "mit-han-lab/opt-125m-smoothquant"
OUTPUT_DIR = Path("/content/quant_outputs/smoothquant_w8a8")
PROMPT = "Quantization reduces inference memory by"
MAX_NEW_TOKENS = 64
RUNS = 3

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)

# Path 1: load the official exported INT8 SmoothQuant model when available.
int8_model = Int8OPTForCausalLM.from_pretrained(SMOOTHQUANT_HF_ID)
int8_model.save_pretrained(OUTPUT_DIR / "official_int8_model")
tokenizer.save_pretrained(OUTPUT_DIR / "official_int8_model")

metrics = {
    "method": "smoothquant_w8a8_official_export",
    "model_id": MODEL_ID,
    "smoothquant_hf_id": SMOOTHQUANT_HF_ID,
    "artifact_dir": str(OUTPUT_DIR / "official_int8_model"),
    "artifact_size_gib": bytes_to_gib(directory_size_bytes(OUTPUT_DIR / "official_int8_model")),
}
save_metrics(OUTPUT_DIR / "metrics_official_export.json", metrics)

In [ ]:
# Path 2: smooth and fake-quantize locally for a quick accuracy sanity check.
# This keeps computation Colab-friendly. Use the official export_int8_model.py script
# for real INT8 export of larger OPT models after downloading the Pile validation file.

device = "cuda" if torch.cuda.is_available() else "cpu"
model = OPTForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)
act_scale_path = Path("act_scales") / "opt-125m.pt"
act_scales = torch.load(act_scale_path, map_location="cpu")
smooth_lm(model, act_scales, alpha=0.5)
fake_w8a8_model = quantize_opt(model)

inputs = tokenizer(PROMPT, return_tensors="pt").to(next(fake_w8a8_model.parameters()).device)
with torch.no_grad():
    out = fake_w8a8_model.generate(**inputs, max_new_tokens=32, do_sample=False)
print(tokenizer.decode(out[0], skip_special_tokens=True))

torch.save(
    {"model_id": MODEL_ID, "alpha": 0.5, "act_scales": act_scales},
    OUTPUT_DIR / "smoothquant_scales_and_config.pt",
)
print("Saved smoothing config and scales:", OUTPUT_DIR / "smoothquant_scales_and_config.pt")

In [ ]:
# Optional official export command for real INT8 OPT artifacts.
# It requires a Pile validation jsonl.zst file. Uncomment after placing the file.
#
# !python examples/export_int8_model.py \
#   --model-name facebook/opt-125m \
#   --act-scales act_scales/opt-125m.pt \
#   --output-path /content/quant_outputs/smoothquant_export \
#   --dataset-path /content/val.jsonl.zst \
#   --num-samples 128 \
#   --seq-len 512